mengimport library dan model yang digunakan

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

membaca dataset

In [ ]:
data = pd.read_csv("dataset_radiasi_surya_indonesia.csv")

nama_kolom_baru = {
    "date": "tanggal",
    "ghi_kwh_m2_day": "radiasi_matahari",
    "temp_2m_c": "suhu_udara_c",
    "humidity_2m_pct": "kelembapan_persen",
    "wind_2m_ms": "kecepatan_angin_ms",
    "rain_mm_day": "curah_hujan_mm",
    "cloud_pct": "tutupan_awan_persen",
    "location": "kota",
    "latitude": "lintang",
    "longitude": "bujur",
    "day_of_year": "hari_ke",
    "month": "bulan",
}
data = data.rename(columns=nama_kolom_baru)

KOLOM_TARGET = "radiasi_matahari"
data["tanggal"] = pd.to_datetime(data["tanggal"], errors="raise")

print("Sumber data: NASA POWER Daily API")
print("Ukuran dataset:", data.shape)
print(data.head())


cleaning dan preprocesing dataa

In [ ]:
data.columns = data.columns.str.strip()
data = data.replace(-999, np.nan)
data["tanggal"] = pd.to_datetime(data["tanggal"], errors="coerce")
data = data.drop_duplicates(["kota", "tanggal"])

kolom_wajib = [
    "tanggal", "kota", "lintang", "bujur", KOLOM_TARGET,
    "suhu_udara_c", "kelembapan_persen", "kecepatan_angin_ms", "tutupan_awan_persen"
]

data = data.dropna(subset=kolom_wajib).copy()

print("Ukuran setelah cleaning:", data.shape)
print("Jumlah kota:", data["kota"].nunique())
print("Rentang tanggal:", data["tanggal"].min().date(), "s.d.", data["tanggal"].max().date())
print("Data kosong:\n", data[kolom_wajib].isna().sum())
print("Data hujan di atas 100 mm/hari:", (data["curah_hujan_mm"] > 100).sum())

membuat grafik visual GHI

In [ ]:
gambar, sumbu = plt.subplots(1, 2, figsize=(12, 4))

data[KOLOM_TARGET].hist(bins=30, ax=sumbu[0])
sumbu[0].set_title("Sebaran GHI")
sumbu[0].set_xlabel("GHI (kWh/m²/hari)")

data.groupby(data["tanggal"].dt.month)[KOLOM_TARGET].mean().plot(ax=sumbu[1], marker="o")
sumbu[1].set_title("Rata-rata GHI per Bulan")
sumbu[1].set_xlabel("Bulan")
sumbu[1].set_ylabel("GHI (kWh/m²/hari)")

plt.tight_layout()
plt.show()

print(data.groupby("kota")[KOLOM_TARGET].mean().sort_values(ascending=False))

menambahkan fitur 

In [ ]:
data["bulan"] = data["tanggal"].dt.month
hari_dalam_tahun = data["tanggal"].dt.dayofyear

data["musim_sin"] = np.sin(2 * np.pi * hari_dalam_tahun / 365.25)
data["musim_cos"] = np.cos(2 * np.pi * hari_dalam_tahun / 365.25)

DAFTAR_FITUR = [
    "suhu_udara_c",
    "kelembapan_persen",
    "kecepatan_angin_ms",
    "tutupan_awan_persen",
    "lintang",
    "bujur",
    "musim_sin",
    "musim_cos"
]

print("Target:", KOLOM_TARGET)
print("Fitur:", DAFTAR_FITUR)

membagi data latih dan data testing

In [ ]:
data_latih = data[data["tanggal"] < "2023-01-01"]
data_uji = data[data["tanggal"] >= "2023-01-01"]

X_latih = data_latih[DAFTAR_FITUR]
y_latih = data_latih[KOLOM_TARGET]

X_uji = data_uji[DAFTAR_FITUR]
y_uji = data_uji[KOLOM_TARGET]

print("Data training:", len(data_latih), "baris")
print("Periode training:", data_latih["tanggal"].min().date(), "s.d.", data_latih["tanggal"].max().date())
print("Data testing:", len(data_uji), "baris")
print("Periode testing:", data_uji["tanggal"].min().date(), "s.d.", data_uji["tanggal"].max().date())

memuat model random forest regresor

In [ ]:
model_rf = RandomForestRegressor(
    n_estimators=200,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

model_rf.fit(X_latih, y_latih)

prediksi_uji = np.maximum(model_rf.predict(X_uji), 0)

print("Model Random Forest selesai dilatih.")

mengevaluasi model

In [ ]:
nilai_mae = mean_absolute_error(y_uji, prediksi_uji)
nilai_rmse = np.sqrt(mean_squared_error(y_uji, prediksi_uji))
nilai_r2 = r2_score(y_uji, prediksi_uji)

print(f"MAE:  {nilai_mae:.3f} kWh/m²/hari")
print(f"RMSE: {nilai_rmse:.3f} kWh/m²/hari")
print(f"R²:   {nilai_r2:.3f}")